# Chapter 9: Decorators and Closures
*From: Fluent Python by Luciano Ramalho*

---

Function decorators let us "mark" functions to enhance their behavior. Mastering decorators requires understanding **closures** -- when functions capture variables from enclosing scopes. The `nonlocal` keyword completes the picture by enabling inner functions to rebind captured variables.

## TL;DR

- A **decorator** is a callable that takes a function and returns a replacement (or the same) function. `@decorator` is sugar for `func = decorator(func)`.
- Decorators run at **import time**, not when the decorated function is called.
- Python determines variable scope at **compile time**: assignment makes a name local.
- A **closure** captures free variables from the enclosing scope via `__closure__`, letting inner functions remember state.
- **`nonlocal`** lets an inner function rebind immutable free variables (numbers, strings) in the enclosing scope.
- A proper decorator uses `@functools.wraps` to preserve the original function's metadata.
- **`functools.cache`** and **`lru_cache`** provide automatic memoization.
- **`functools.singledispatch`** enables type-based dispatch without if/elif chains.
- **Parameterized decorators** require a decorator factory: factory -> decorator -> wrapper.

---
## 1. Decorator Fundamentals

A decorator is a callable that takes a function as an argument and returns a replacement (or the same) function.

```
@decorate
def target():
    ...

# is exactly equivalent to:
target = decorate(target)
```

**Three essential facts:**
1. A decorator is a function or another callable
2. A decorator may replace the decorated function with a different one
3. Decorators are executed immediately when a module is loaded (import time)

In [ ]:
# A decorator usually replaces a function with a different one
def deco(func):
    def inner():
        print("running inner()")
    return inner

@deco
def target():
    print("running target()")

target()  # Prints "running inner()" -- target was replaced!
print(f"target is now: {target.__name__}")

In [ ]:
# Registration decorator: runs at "import time"
registry = []

def register(func):
    print(f"  register -> {func.__name__}")
    registry.append(func)
    return func  # returns the same function, unchanged

@register
def f1():
    print("running f1()")

@register
def f2():
    print("running f2()")

def f3():
    print("running f3()")

print(f"\nregistry: {[f.__name__ for f in registry]}")
# Notice: register() ran at definition time, before we call anything

---
## 2. Variable Scope Rules and Closures

Python determines variable scope at **compile time**. If a variable is assigned anywhere in the function body, it is treated as **local** throughout.

In [ ]:
# Scope surprise: assignment makes b local
b = 6

def f1(a):
    print(a)
    print(b)  # works: b is global

f1(3)

def f2(a):
    print(a)
    # print(b)  # would fail: UnboundLocalError!
    b = 9      # this assignment makes b local for the ENTIRE function

# Uncommenting print(b) in f2 would raise:
# UnboundLocalError: local variable \'b\' referenced before assignment
print("f2 would fail because b is assigned -> local")

In [ ]:
# Closure: inner function captures free variables
def make_averager():
    series = []  # free variable captured by closure

    def averager(new_value):
        series.append(new_value)  # mutating works (list is mutable)
        total = sum(series)
        return total / len(series)

    return averager

avg = make_averager()
print(f"avg(10) = {avg(10)}")
print(f"avg(11) = {avg(11)}")
print(f"avg(15) = {avg(15)}")

# Inspect the closure
print(f"\nFree vars: {avg.__code__.co_freevars}")
print(f"Closure:   {avg.__closure__[0].cell_contents}")

---
## 3. The nonlocal Declaration

When you need to **rebind** an immutable free variable (number, string, tuple), use `nonlocal`. Without it, assignment creates a new local variable.

In [ ]:
# Without nonlocal: BROKEN
def make_averager_broken():
    count = 0
    total = 0
    def averager(new_value):
        # count += 1   # This would fail: UnboundLocalError
        # total += new_value  # Same issue
        pass
    return averager

# With nonlocal: FIXED
def make_averager():
    count = 0
    total = 0

    def averager(new_value):
        nonlocal count, total  # rebind the enclosing variables
        count += 1
        total += new_value
        return total / count

    return averager

avg = make_averager()
print(f"avg(10)  = {avg(10)}")
print(f"avg(11)  = {avg(11)}")
print(f"avg(12)  = {avg(12)}")

---
## 4. Implementing a Well-Behaved Decorator

A proper decorator should:
1. Use an inner wrapper with `*args, **kwargs`
2. Call the original function via the closure
3. Use `@functools.wraps` to preserve the original's metadata

In [ ]:
import time
import functools

def clock(func):
    @functools.wraps(func)  # preserves __name__, __doc__, etc.
    def clocked(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        arg_str = ", ".join(repr(a) for a in args)
        print(f"[{elapsed:0.8f}s] {func.__name__}({arg_str}) -> {result!r}")
        return result
    return clocked

@clock
def snooze(seconds):
    """Sleep for a while."""
    time.sleep(seconds)

@clock
def factorial(n):
    """Return n!"""
    return 1 if n < 2 else n * factorial(n - 1)

snooze(0.05)
print(f"6! = {factorial(6)}")
print(f"\nPreserved name: {factorial.__name__}")
print(f"Preserved doc:  {factorial.__doc__}")

---
## 5. Memoization with functools.cache and lru_cache

`functools.cache` (Python 3.9+) and `functools.lru_cache` store results of previous calls, keyed by arguments.

In [ ]:
import functools

@functools.cache  # unbounded cache (Python 3.9+)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 2) + fibonacci(n - 1)

# Without cache, this would make 2^30 recursive calls!
print(f"fib(30) = {fibonacci(30)}")
print(f"Cache info: {fibonacci.cache_info()}")

In [ ]:
# lru_cache: bounded cache with LRU eviction
@functools.lru_cache(maxsize=128)
def factorial_cached(n):
    return 1 if n < 2 else n * factorial_cached(n - 1)

print(f"20! = {factorial_cached(20)}")
print(f"Cache info: {factorial_cached.cache_info()}")

# The typed parameter (default False) treats 1 and 1.0 differently
@functools.lru_cache(maxsize=32, typed=True)
def add_one(n):
    return n + 1

print(f"\nadd_one(1)   = {add_one(1)}")
print(f"add_one(1.0) = {add_one(1.0)}")
print(f"Cache info:  {add_one.cache_info()}")  # 2 misses (typed=True)

---
## 6. Single Dispatch Generic Functions

`functools.singledispatch` dispatches on the type of the **first argument**, enabling extensible polymorphism without if/elif chains.

In [ ]:
from functools import singledispatch
import numbers

@singledispatch
def htmlize(obj):
    """Default: escape and wrap in <pre>."""
    content = repr(obj)
    return f"<pre>{content}</pre>"

@htmlize.register
def _(text: str) -> str:
    """Strings: replace newlines with <br> and wrap in <p>."""
    inner = text.replace("\n", "<br>\n")
    return f"<p>{inner}</p>"

@htmlize.register
def _(n: numbers.Integral) -> str:
    """Integers: show decimal and hex."""
    return f"<pre>{n} (0x{n:x})</pre>"

@htmlize.register
def _(seq: list) -> str:
    """Lists: render as <ul>."""
    items = "\n".join(f"  <li>{htmlize(item)}</li>" for item in seq)
    return f"<ul>\n{items}\n</ul>"

# Test it
print(htmlize(42))
print(htmlize("Hello\nWorld"))
print(htmlize([1, "two", 3]))
print(htmlize({"a": 1}))  # falls back to default

---
## 7. Parameterized Decorators

A parameterized decorator is a **decorator factory**: a function that accepts configuration arguments and returns the actual decorator. This requires three levels of nesting.

In [ ]:
import functools

# Decorator factory: accepts parameters, returns the actual decorator
def repeat(n=2):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(n=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Alice")  # prints 3 times

In [ ]:
# Class-based decorator: cleaner for complex parameterized decorators
import functools

class Retry:
    """Retry a function up to max_retries times on exception."""
    def __init__(self, max_retries=3):
        self.max_retries = max_retries

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, self.max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  Attempt {attempt} failed: {e}")
            raise last_error
        return wrapper

@Retry(max_retries=3)
def flaky():
    import random
    random.seed(42)
    if random.random() < 0.7:
        raise ValueError("bad luck")
    return "success!"

try:
    print(flaky())
except ValueError as e:
    print(f"All retries failed: {e}")

---
## Try It Yourself

### Exercise 1: Timing Decorator with Threshold
Write a decorator `warn_slow(threshold)` that prints a warning if the decorated function takes longer than `threshold` seconds.

In [ ]:
import time
import functools

def warn_slow(threshold=1.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - t0
            if elapsed > threshold:
                print(f"WARNING: {func.__name__} took {elapsed:.3f}s ")
            return result
        return wrapper
    return decorator

@warn_slow(threshold=0.05)
def fast():
    time.sleep(0.01)
    return "done fast"

@warn_slow(threshold=0.05)
def slow():
    time.sleep(0.1)
    return "done slow"

print(fast())  # no warning
print(slow())  # prints warning

### Exercise 2: Memoize with nonlocal
Implement a simple memoization closure (not a decorator) that caches results of an expensive function. Use `nonlocal` for the cache dict.

In [ ]:
# Exercise 2
def make_memoized(func):
    cache = {}

    def memoized(*args):
        if args not in cache:
            cache[args] = func(*args)
            print(f"  computed {func.__name__}{args}")
        else:
            print(f"  cached   {func.__name__}{args}")
        return cache[args]

    return memoized

def expensive_square(n):
    return n ** 2

memo_sq = make_memoized(expensive_square)
print(memo_sq(4))
print(memo_sq(4))  # cached
print(memo_sq(5))
print(memo_sq(5))  # cached

### Exercise 3: singledispatch for format_value
Create a singledispatch function `format_value` that formats int as hex, float to 2 decimal places, str in quotes, and everything else with repr.

In [ ]:
from functools import singledispatch

@singledispatch
def format_value(val):
    return repr(val)

@format_value.register
def _(val: int) -> str:
    return f"0x{val:x}"

@format_value.register
def _(val: float) -> str:
    return f"{val:.2f}"

@format_value.register
def _(val: str) -> str:
    return f'"{val}"'

print(format_value(255))       # 0xff
print(format_value(3.14159))   # 3.14
print(format_value("hello"))   # "hello"
print(format_value([1, 2]))    # [1, 2]

---
## Key Takeaways

1. **Decorators run at import time**, not at call time. The `@decorator` syntax is sugar for `func = decorator(func)`.

2. **Python determines scope at compile time**: assigning to a variable makes it local. Use `global` or `nonlocal` to override.

3. A **closure** is a function that captures free variables from the enclosing scope, allowing it to "remember" state after the enclosing function returns.

4. **`nonlocal`** is essential when an inner function needs to rebind an immutable variable (int, str, tuple) in the enclosing scope.

5. **Always use `@functools.wraps`** in decorator implementations to preserve the original function's `__name__`, `__doc__`, and other attributes.

6. **`@functools.cache`** and **`@lru_cache`** provide effortless memoization for pure functions.

7. **`@singledispatch`** replaces `if isinstance()` chains with clean, extensible type-based dispatch.

8. **Parameterized decorators** require a factory function that returns the actual decorator (three nesting levels).

---
## See Also

- [[decorator-fundamentals]] -- What decorators are and import-time execution
- [[variable-scope-and-closures]] -- Scope rules, free variables, and closures
- [[nonlocal-declaration]] -- Rebinding captured variables
- [[implementing-decorators]] -- Building decorators with @functools.wraps
- [[functools-cache-lru-cache]] -- Memoization decorators
- [[singledispatch-generic-functions]] -- Type-based dispatch
- [[parameterized-decorators]] -- Decorator factories and class-based decorators

**Next:** Chapter 10 applies first-class functions to the Strategy design pattern.